# Geofence circle classifier (synthetic)

Predict whether a rider point lies inside a circular disruption/policy zone. Labels are generated with India hub centers and radii aligned with `SINGLE_ZONE_RADIUS_KM` / `DEFAULT_GEOFENCE_RADIUS_KM` in [oasis_constants.json](../oasis_constants.json). Add small GPS noise near the boundary to mimic real fixes.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
from synthetic_data import generate_geofence_rows, write_datasets
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

df = generate_geofence_rows(8000)
df.head()

In [ ]:
# No dist_km / deltas — they leak the label (distance to center encodes inside/outside).
features = ["center_lat", "center_lng", "radius_km", "rider_lat", "rider_lng"]
X, y = df[features], df["inside_zone"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=80, max_depth=10, random_state=42, class_weight="balanced")),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print("holdout accuracy", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

In [ ]:
import joblib
ROOT.joinpath("artifacts").mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, ROOT / "artifacts" / "geofence_circle.joblib")
df.to_csv(ROOT / "data" / "synthetic" / "geofence_circle.csv", index=False)
print("Saved artifacts/geofence_circle.joblib and data/synthetic/geofence_circle.csv")